<a href="https://colab.research.google.com/github/bsong75/brendensong.github.io/blob/main/Neo4j_FASTRP_emb_optimize.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install neo4j
!pip install optuna

     |████████████████████████████████| 76 kB 1.6 MB/s 
  Created wheel for neo4j: filename=neo4j-4.3.7-py3-none-any.whl size=100642 sha256=1e4582d280d06ad6bef16df28b85855c687e71f312746625bb0f00efbe27e9a8
  Stored in directory: /root/.cache/pip/wheels/b5/24/bb/cece9fcfdd5e1aa0683e2533945e1e3f27f70f342ff7e28993
Successfully built neo4j
     |████████████████████████████████| 308 kB 2.7 MB/s 
     |████████████████████████████████| 209 kB 49.4 MB/s 
     |████████████████████████████████| 80 kB 8.7 MB/s 
     |████████████████████████████████| 75 kB 3.5 MB/s 
     |████████████████████████████████| 144 kB 53.5 MB/s 
     |████████████████████████████████| 49 kB 5.6 MB/s 
     |████████████████████████████████| 111 kB 56.7 MB/s 
  Created wheel for pyperclip: filename=pyperclip-1.8.2-py3-none-any.whl size=11136 sha256=0d898a3576c60a07b0099f6a4d50b402618420b57ec44fe08dfecaae98d37f88
  Stored in directory: /root/.cache/pip/wheels/9f/18/84/8f69f8b08169c7bae2dde6bd7daf0c19fca8c8e500ee620a28
S

In [ ]:
%matplotlib inline

import ast

from neo4j import GraphDatabase

import numpy as np
import pandas as pd

import optuna

from sklearn.manifold import TSNE
from sklearn import svm
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import plot_confusion_matrix

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns


In [ ]:
class Neo4jConnection:

    def __init__(self, uri, user, pwd):
        self.__uri = uri
        self.__user = user
        self.__pwd = pwd
        self.__driver = None
        try:
            self.__driver = GraphDatabase.driver(self.__uri, auth=(self.__user, self.__pwd))
        except Exception as e:
            print("Failed to create the driver:", e)

    def close(self):
        if self.__driver is not None:
            self.__driver.close()

    def query(self, query, parameters=None, db=None):
        assert self.__driver is not None, "Driver not initialized!"
        session = None
        response = None
        try:
            session = self.__driver.session(database=db) if db is not None else self.__driver.session()
            response = list(session.run(query, parameters))
        except Exception as e:
            print("Query failed:", e)
        finally:
            if session is not None:
                session.close()
        return response

In [ ]:
uri = 'bolt://44.192.100.236:7687'
pwd = 'shout-holddowns-temper'

conn = Neo4jConnection(uri=uri, user="neo4j", pwd=pwd)

conn.query("MATCH (n) RETURN COUNT(n)")

[<Record COUNT(n)=2708>]

In [ ]:
def create_X_y():

    query = """MATCH (p:Paper) RETURN p.id AS id, p.subject AS subject, p.fastrp_embedding AS fastrp_embedding"""
    emb_df = pd.DataFrame([dict(_) for _ in conn.query(query)])
    emb_df['target'] = pd.factorize(emb_df['subject'])[0].astype("float32")
    y = emb_df['target'].to_numpy()
    emb_df['X'] = emb_df['fastrp_embedding'].apply(lambda x: np.array(x))
    X = np.array(emb_df['X'].to_list())

    return X, y


def modeler(params):

    acc_scores = []
    k_folds = 5

    # create_embs(dim=dim)
    X, y = create_X_y()

    for i in range(0, k_folds):

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25)
        clf = svm.SVC(kernel='linear', class_weight='balanced')
        clf.fit(X_train, y_train)
        pred = clf.predict(X_test)

        acc = accuracy_score(pred, y_test)
        acc_scores.append(acc)

    return np.mean(acc_scores)

In [ ]:
def objectiveSKLearn():

    def objective(params):

        dim = params['embeddingDimension']
        thirdWeight = params['thirdWeight']
        fourthWeight = params['fourthWeight']
        normalizationStrength = params['normalizationStrength']

        # Create embeddings
        query = """CALL gds.fastRP.write(
                    'cora',
                    {
                        embeddingDimension: %d,
                        iterationWeights: [0.0, 0.0, %f, %f],
                        normalizationStrength: %f,
                        randomSeed: 42,
                        writeProperty: 'fastrp_embedding'
                    }
                )
        """ % (dim, thirdWeight, fourthWeight, normalizationStrength)

        conn.query(query)

        return modeler(params)

    return objective

In [ ]:
def optuna_objective(trial):

    params = {
        'embeddingDimension': trial.suggest_int('embeddingDimension', 32, 512, log=True),
        'thirdWeight': trial.suggest_float('thirdWeight', 0.0, 1.0),
        'fourthWeight': trial.suggest_float('fourthWeight', 0.0, 1.0),
        'normalizationStrength': trial.suggest_float('normalizationStrength', -1.0, 1.0)
    }

    return objective(params)


In [ ]:
objective = objectiveSKLearn()

initial_params = {
    'embeddingDimension': 64,
    'thirdWeight': 0.5,
    'fourthWeight': 1.0,
    'normalizationStrength': -0.5

}

study = optuna.create_study(direction='maximize')
study.enqueue_trial(initial_params)
study.optimize(optuna_objective, n_trials=100)

[I 2021-11-02 20:43:23,085] A new study created in memory with name: no-name-d452eda5-0d2c-46a7-9a1c-a16277261844
/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:12: ExperimentalWarning:

enqueue_trial is experimental (supported from v1.2.0). The interface can change in the future.

/usr/local/lib/python3.7/dist-packages/optuna/study/study.py:857: ExperimentalWarning:

create_trial is experimental (supported from v2.0.0). The interface can change in the future.

/usr/local/lib/python3.7/dist-packages/optuna/study/study.py:857: ExperimentalWarning:

add_trial is experimental (supported from v2.0.0). The interface can change in the future.

/usr/local/lib/python3.7/dist-packages/optuna/distributions.py:364: FutureWarning:

Samplers and other components in Optuna will assume that `step` is 1. `step` argument is deprecated and will be removed in the future. The removal of this feature is currently scheduled for v4.0.0, but this schedule is subject to change.

[I 2021-11-02 20:

In [ ]:
fig = optuna.visualization.plot_optimization_history(study)
fig.show()

In [ ]:
print('Accuracy: {}'.format(study.best_trial.value))
print("Best hyperparameters: {}".format(study.best_trial.params))

Accuracy: 0.869423929098966
Best hyperparameters: {'embeddingDimension': 482, 'thirdWeight': 0.9034019833656333, 'fourthWeight': 0.9708537338500505, 'normalizationStrength': 0.8040817386374997}


In [ ]:
fig = optuna.visualization.plot_param_importances(study)
fig.show()